# Indicador de Seguimiento a la Economía (ISE): pronóstico a 6 meses con la serie diferenciada y análisis de tendencia
## Árbol de decisión, MLP y LSTM, paso a paso

**Objetivo.** Pronosticar el Indicador de Seguimiento a la Economía (ISE) mensual de Colombia **6 meses adelante** con tres modelos de aprendizaje automático (árbol de decisión, perceptrón multicapa y red LSTM), modelando la **serie diferenciada** $\Delta Y_t = Y_t - Y_{t-1}$.

**Novedad en la exploración.** Se incluye una evaluación formal de **6 técnicas distintas de estimación de tendencia** (filtro HP, polinomios, STL, splines, LOESS y diferenciación) para comparar cómo extraen la señal de largo plazo y justificar el uso de la serie diferenciada en los modelos sin fuga de información (*data leakage*).

| Sección | Contenido |
|---|---|
| 1. Preparación | Librerías, datos y fechas del ISE |
| 2. Exploración y tendencias | **Estimación de tendencia por 6 técnicas**, dependencia y estacionalidad |
| 3. El problema de pronóstico | Notación, suma telescópica para volver a niveles, las cuatro estrategias y persistencia |
| 4. De la serie a la tabla | Matriz supervisada, partición temporal con "gap" (hueco), ventanas y motor de búsqueda |
| 5. Árbol de decisión | Inputs, curva de validación, búsqueda y evaluación |
| 6. MLP | Qué cambia frente al árbol, modelo elemental, búsqueda reducida y evaluación |
| 7. LSTM | Secuencias 3D, modelo elemental, búsqueda y evaluación |
| 8. Comparación final | Elección con validación, error en prueba y desglose por horizonte |
| 9. Producción | Pronóstico fuera de muestra de **julio a diciembre de 2026** |
| 10. Conclusiones | Síntesis de resultados |
| 11. Pendientes | Tareas prácticas |

# 1. Preparación

In [ ]:
# %%
import os
import time
import random
import pickle
import itertools
import warnings
import logging
from pathlib import Path

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.interpolate import UnivariateSpline

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, month_plot
from statsmodels.nonparametric.smoothers_lowess import lowess

# Importación robusta del filtro Hodrick-Prescott
try:
    from statsmodels.tsa.filters.hp_filter import hpfilter
except (ImportError, ModuleNotFoundError):
    import statsmodels.api as sm
    hpfilter = sm.tsa.filters.hpfilter

from statsmodels.tsa.seasonal import STL

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

import keras

logging.getLogger('tensorflow').setLevel(logging.ERROR)
plt.style.use('ggplot')
SEMILLA = 123
random.seed(SEMILLA)
np.random.seed(SEMILLA)
keras.utils.set_random_seed(SEMILLA)

print("keras", keras.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

ImportError: cannot import name 'hpfilter' from 'statsmodels.tsa.api' (/home/carlos/Documents/Github/Series-de-tiempo-2026-2/venv/lib/python3.12/site-packages/statsmodels/tsa/api.py)

**Carga de los datos del ISE.** Localizamos el archivo `ISE2005_2026.xlsx` en la carpeta `Bases_de_datos`:

In [ ]:
DIRECTORIO_ACTUAL = Path.cwd()
RUTA_DATOS = DIRECTORIO_ACTUAL.parent / "Bases_de_datos" / "ISE2005_2026.xlsx"
if not RUTA_DATOS.exists():
    RUTA_DATOS = Path("Bases_de_datos") / "ISE2005_2026.xlsx"

datos = pd.read_excel(RUTA_DATOS, parse_dates=[0])
Y = datos['Iset'].to_numpy(dtype=float)
FECHAS = pd.DatetimeIndex(datos['Fecha'])
N = len(Y)

print(f"Ruta cargada: {RUTA_DATOS}")
print(f"Observaciones: {N}, desde {FECHAS[0]:%Y-%m} hasta {FECHAS[-1]:%Y-%m}")
datos.head()

# 2. Exploración y Estimación de la Tendencia

## 2.1 Estimación de la tendencia por múltiples técnicas

Para extraer y modelar el comportamiento de largo plazo del ISE implementamos y comparamos **6 técnicas**:
1. **Diferenciación:** $\Delta Y_t = Y_t - Y_{t-1}$ (elimina la raíz unitaria sin estimación de parámetros).
2. **Filtro Hodrick-Prescott (HP):** Descompone $Y_t = \tau_t + c_t$. En datos mensuales se utiliza $\lambda = 14\,400$.
3. **Tendencia Polinomial (Grado 1 y 2):** Ajuste por mínimos cuadrados ordinarios ($T_t = \beta_0 + \beta_1 t + \beta_2 t^2$).
4. **LOESS:** Regresión local no paramétrica ponderada.
5. **Splines Cúbicos de Suavizamiento:** Ajuste por polinomios a trozos con penalización de curvatura (`UnivariateSpline`).
6. **Descomposición STL:** Descomposición estacional y de tendencia mediante LOESS.

In [ ]:
t_idx = np.arange(N)

# 1. Filtro Hodrick-Prescott (lambda = 14400 para frecuencia mensual)
hp_ciclo, hp_tendencia = hpfilter(Y, lamb=14400)

# 2. Tendencia Lineal y Cuadrática (OLS)
poly_1 = PolynomialFeatures(degree=1)
X_p1 = poly_1.fit_transform(t_idx.reshape(-1, 1))
tend_lineal = LinearRegression().fit(X_p1, Y).predict(X_p1)

poly_2 = PolynomialFeatures(degree=2)
X_p2 = poly_2.fit_transform(t_idx.reshape(-1, 1))
tend_cuadratica = LinearRegression().fit(X_p2, Y).predict(X_p2)

# 3. LOESS
tend_loess = lowess(Y, t_idx, frac=0.25, return_sorted=False)

# 4. Splines Cúbicos de Suavizamiento
spline = UnivariateSpline(t_idx, Y, s=len(Y) * 2.0)
tend_spline = spline(t_idx)

# 5. Descomposición STL
res_stl = STL(pd.Series(Y, index=FECHAS), seasonal=13).fit()
tend_stl = res_stl.trend.values

In [ ]:
# Gráfico comparativo de todas las tendencias estimadas
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(FECHAS, Y, color='black', alpha=0.35, label='ISE observado ($Y_t$)', lw=1)
ax.plot(FECHAS, tend_lineal, '--', color='#7f8c8d', lw=1.5, label='Lineal OLS')
ax.plot(FECHAS, tend_cuadratica, '-.', color='#8e44ad', lw=1.8, label='Polinomial Grado 2')
ax.plot(FECHAS, hp_tendencia, '-', color='#2980b9', lw=2, label='Filtro HP ($\lambda=14400$)')
ax.plot(FECHAS, tend_loess, '-', color='#e67e22', lw=2, label='LOESS (frac=0.25)')
ax.plot(FECHAS, tend_spline, ':', color='#27ae60', lw=2.2, label='Spline Cúbico')
ax.plot(FECHAS, tend_stl, '-', color='#c0392b', lw=1.8, label='STL (Loess)')

ax.set_title("Comparación de 6 técnicas de estimación de tendencia en el ISE", fontweight='bold')
ax.set_ylabel("Puntos del ISE")
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa de residuos y propiedades
tabla_tendencias = pd.DataFrame({
    'Lineal': Y - tend_lineal,
    'Cuadrática': Y - tend_cuadratica,
    'Filtro HP': Y - hp_tendencia,
    'LOESS': Y - tend_loess,
    'Spline': Y - tend_spline,
    'STL': Y - tend_stl
}, index=FECHAS)

resumen_tendencias = pd.DataFrame({
    'Desv. Est. Residuo': tabla_tendencias.std(),
    'EAM Residuo': tabla_tendencias.abs().mean(),
    'Mínimo Residuo': tabla_tendencias.min(),
    'Máximo Residuo': tabla_tendencias.max(),
    'Riesgo Fuga en Pronóstico': [
        'Bajo (proyección analítica)',
        'Moderado (proyección analítica)',
        'Alto (efecto de borde / sin extrapolación)',
        'Alto (no proyecta fuera de muestra)',
        'Alto (extrapolación inestable en bordes)',
        'Alto (requiere modelo secundario para T)'
    ]
})
print("Propiedades de las técnicas de tendencia aplicadas al ISE:")
display(resumen_tendencias.round(3))

**Conclusión sobre la tendencia para Machine Learning:**
- Métodos como **Splines, LOESS o Filtro HP** logran un ajuste histórico suave, pero presentan **severos efectos de borde** y no pueden proyectarse fuera de muestra sin recurrir a supuestos ad hoc (riesgo de fuga o error de extrapolación).
- La **diferenciación de primer orden** ($\Delta Y_t = Y_t - Y_{t-1}$) transforma la serie en estacionaria en media sin estimar ningún parámetro, eliminando por construcción cualquier riesgo de fuga de información.

In [ ]:
Z_serie = pd.Series(np.diff(Y), index=FECHAS[1:], name='ΔY')

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(FECHAS, Y, color='black', lw=1, label='ISE ($Y_t$)')
axes[0].plot(FECHAS, hp_tendencia, color='red', lw=1.8, label='Tendencia Filtro HP')
axes[0].set_ylabel("Puntos ISE")
axes[0].set_title("Serie original del ISE con tendencia y choque del 2020", fontweight='bold')
axes[0].legend(fontsize=8)

axes[1].plot(Z_serie.index, Z_serie.values, color='#c0392b', lw=1)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_ylabel("Δ ISE")
axes[1].set_title(r"Serie diferenciada $\Delta Y_t = Y_t - Y_{t-1}$: oscila alrededor de cero", fontweight='bold')
plt.tight_layout()
plt.show()

## 2.2 Dependencia de la serie diferenciada

In [ ]:
rezagos_graficar = [1, 2, 3, 12]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, k in zip(axes, rezagos_graficar):
    x, y_val = Z_serie.values[:-k], Z_serie.values[k:]
    ax.scatter(x, y_val, s=10, alpha=0.5, color='darkblue')
    curva = lowess(y_val, x, frac=0.67)
    ax.plot(curva[:, 0], curva[:, 1], color='red', lw=2)
    ax.set_title(f"Rezago {k}", fontweight='bold')
    ax.set_xlabel(rf"$\Delta Y_{{t-{k}}}$")
axes[0].set_ylabel(r"$\Delta Y_t$")
plt.suptitle("Gráficos de dispersión LOESS de rezagos sobre la serie diferenciada", fontweight='bold')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
plot_acf(Z_serie, lags=36, ax=axes[0], title="ACF de la serie diferenciada")
plot_pacf(Z_serie, lags=36, ax=axes[1], title="PACF de la serie diferenciada", method='ywm')
plt.tight_layout()
plt.show()

In [ ]:
z = Z_serie.values
try:
    import dcor
    dcor_rezagos = pd.Series([dcor.distance_correlation(z[k:], z[:-k]) for k in range(1, 37)],
                             index=pd.Index(range(1, 37), name='rezago'), name='dCor')
    ax = dcor_rezagos.plot.bar(figsize=(11, 3), color='#8e44ad')
    ax.set_title("Correlación de distancia (dCor) entre $\\Delta Y_t$ y $\\Delta Y_{t-k}$", fontweight='bold')
    ax.set_ylabel("dCor"); plt.tight_layout(); plt.show()
    print("dCor rezagos 1, 2, 3 y 12:", dcor_rezagos.loc[[1, 2, 3, 12]].round(3).to_dict())
except ImportError:
    print("Librería dcor no instalada; se continúa con ACF y PACF.")

## 2.3 Estacionalidad de la serie diferenciada

In [ ]:
meses_es = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
tabla_mes = pd.DataFrame({'ΔY': Z_serie.values, 'Mes': pd.Categorical(
    [meses_es[m - 1] for m in Z_serie.index.month], categories=meses_es, ordered=True)})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=tabla_mes, x='Mes', y='ΔY', hue='Mes', legend=False, palette='mako', ax=axes[0])
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title("Cambio mensual del ISE por mes del año", fontweight='bold')

month_plot(Z_serie, ax=axes[1])
axes[1].set_xticks(range(12), meses_es)
axes[1].set_title("Subseries mensuales de ΔY", fontweight='bold')
plt.tight_layout()
plt.show()

frecuencias, potencia = signal.periodogram(z)
plt.figure(figsize=(10, 3.2))
plt.plot(frecuencias, potencia, color='#2c3e50')
for periodo in [12, 6, 4, 3]:
    plt.axvline(1 / periodo, color='darkred', ls='--', lw=1)
    plt.text(1 / periodo, potencia.max() * 0.9, f" {periodo}m", color='darkred', fontsize=8)
plt.title("Periodograma de la serie diferenciada del ISE", fontweight='bold')
plt.xlabel("Frecuencia (ciclos por mes)")
plt.tight_layout()
plt.show()

# 3. El problema de pronóstico

## 3.1 Notación y suma telescópica
Para pronosticar el **nivel** $Y_{t+6}$ acumulamos los seis cambios predichos a partir del origen conocido $Y_t$:

$$Y_{t+6} = Y_t + \sum_{j=1}^{6} Z_{t+j}$$

## 3.2 Las cuatro estrategias evaluadas
1. **Recursiva:** Un modelo de un paso reutilizado 6 veces deslizando la ventana.
2. **Directa a $h=6$:** Un modelo entrenado específicamente para el cambio acumulado $Y_{t+6} - Y_t$.
3. **Directa con 6 modelos:** 6 modelos independientes, cada uno aprende $Z_{t+h}$.
4. **MIMO (*one-shot*):** Un único modelo con vector de salida de 6 dimensiones $(Z_{t+1}, \dots, Z_{t+6})$.

La referencia base es la **persistencia** (pronóstico ingenuo): $\hat{Y}_{t+h} = Y_t$.

# 4. De la serie a la tabla supervisada

In [ ]:
H = 6
VENTANA = 13
LAGS_MODELO = [1, 2, 3, 4, 13]  # Convención: lag_1 es t-1, lag_13 es t-13

FECHAS_EXT = pd.date_range(start=FECHAS[0], periods=N + 1, freq=FECHAS.freq or 'MS')
dY = pd.Series(np.diff(Y), index=FECHAS[1:]).reindex(FECHAS_EXT)
nivel = pd.Series(Y, index=FECHAS).reindex(FECHAS_EXT)
mes = pd.Series(FECHAS_EXT.month, index=FECHAS_EXT)

columnas = {}
for k in range(VENTANA, 0, -1):
    columnas[f"lag_{k}"] = dY.shift(k)
    columnas[f"mes_lag_{k}"] = mes.shift(k)

datos = pd.DataFrame(columnas)
for m in range(2, 13):
    datos[f"mes_{m}"] = (datos["mes_lag_1"] == m).astype(float)

for h in range(1, H + 1):
    datos[f"y_h{h}"] = dY.shift(-(h - 1))

datos["y_acum6"] = nivel.shift(-(H - 1)) - nivel.shift(1)
datos["nivel_origen"] = nivel.shift(1)

# Se eliminan los primeros registros sin ventana completa
datos = datos.iloc[VENTANA + 1:]

COLS_TABLA = [f"lag_{k}" for k in LAGS_MODELO] + [f"mes_{m}" for m in range(2, 13)]
COLS_VENTANA = [c for k in range(VENTANA, 0, -1) for c in (f"lag_{k}", f"mes_lag_{k}")]
Y_COLS = [f"y_h{h}" for h in range(1, H + 1)]

print(f"Filas listas: {len(datos)} | De {datos.index[0]:%Y-%m} a {datos.index[-1]:%Y-%m}")
datos[["lag_13", "lag_1", "mes_lag_1", "y_h1", "y_h6", "y_acum6", "nivel_origen"]].tail().round(2)

In [ ]:
RESPUESTAS = {
    'recursiva': ['y_h1'],
    'directa_h': ['y_acum6'],
    'directa_multi': Y_COLS,
    'mimo': Y_COLS
}
ESTRATEGIAS = list(RESPUESTAS)

## 4.2 Partición temporal sin fuga de información (con "gap")

In [ ]:
OFFSET_MES = pd.DateOffset(months=1)
CORTE_TEST = FECHAS[int(N * 0.80)]
CORTE_VAL = FECHAS[int(N * 0.70)]
N_PLIEGUES, TAM_PLIEGUE = 3, 20
CORTES_CV = [CORTE_TEST - pd.DateOffset(months=TAM_PLIEGUE * (N_PLIEGUES - k)) for k in range(N_PLIEGUES)]
ESQUEMAS_CV = {
    'simple': [(CORTE_VAL, CORTE_TEST)],
    'secuencial': [(c, c + pd.DateOffset(months=TAM_PLIEGUE)) for c in CORTES_CV]
}

def filas_entrenamiento(corte, estrategia):
    meses_respuesta = 1 if estrategia == 'recursiva' else H
    ultimo_mes = datos.index + pd.DateOffset(months=meses_respuesta - 1)
    return np.flatnonzero(ultimo_mes < corte)

def filas_evaluacion(inicio, fin):
    mes_6 = datos.index + pd.DateOffset(months=H - 1)
    return np.flatnonzero((datos.index >= inicio) & (mes_6 < fin))

def pliegues(esquema, estrategia):
    return [(filas_entrenamiento(inicio, estrategia), filas_evaluacion(inicio, fin))
            for inicio, fin in ESQUEMAS_CV[esquema]]

FILAS_PRUEBA = filas_evaluacion(CORTE_TEST, FECHAS_EXT[-1])
print(f"Número de filas de prueba: {len(FILAS_PRUEBA)}")

## 4.3 Estandarización y funciones de ventana

In [ ]:
MEDIA = dY[dY.index < CORTES_CV[0]].mean()
DE = dY[dY.index < CORTES_CV[0]].std()
print(f"Media de ΔY (train inicial): {MEDIA:.3f} | Desv. Est.: {DE:.3f}")

def a_z(x):
    return (np.asarray(x) - MEDIA) / DE

def a_puntos(z):
    return np.asarray(z) * DE + MEDIA

def a_ventanas(X):
    return X.reshape(len(X), VENTANA, 2)

W_TODO = a_ventanas(datos[COLS_VENTANA].to_numpy())

def ventana_siguiente(W, z_nuevo):
    mes_nuevo = W[:, -1, 1] % 12 + 1
    paso_nuevo = np.column_stack([z_nuevo, mes_nuevo])[:, None, :]
    return np.concatenate([W[:, 1:, :], paso_nuevo], axis=1)

def pronostico_recursivo(predecir, W, entrada):
    pred = []
    for k in range(H):
        z = predecir(entrada(W)).ravel()
        pred.append(z)
        W = ventana_siguiente(W, z)
    return np.column_stack(pred)

def a_niveles(pred, filas, estrategia):
    origen = datos["nivel_origen"].to_numpy()[filas]
    if estrategia == 'directa_h':
        P = np.full((len(filas), H), np.nan)
        P[:, H - 1] = origen + pred[:, 0]
        return P
    return origen[:, None] + np.cumsum(pred, axis=1)

def niveles_reales(filas):
    return datos["nivel_origen"].to_numpy()[filas, None] + np.cumsum(datos[Y_COLS].to_numpy()[filas], axis=1)

def evaluar(y_real, y_pred):
    return {"RECM": root_mean_squared_error(y_real, y_pred), "EAM": mean_absolute_error(y_real, y_pred)}

def recm_h6(P, filas):
    return evaluar(niveles_reales(filas)[:, H - 1], P[:, H - 1])["RECM"]

def persistencia(filas):
    return np.repeat(datos["nivel_origen"].to_numpy()[filas, None], H, axis=1)

P_PERSISTENCIA = persistencia(FILAS_PRUEBA)
RECM_PERSISTENCIA = recm_h6(P_PERSISTENCIA, FILAS_PRUEBA)
print(f"Persistencia a 6 meses en Prueba: RECM = {RECM_PERSISTENCIA:.4f} puntos")

## 4.4 Motor genérico de ajuste, validación y búsqueda

In [ ]:
def pronosticar(entrada, ajustar, params, estrategia, filas_tr, filas_ev, filas_va=None, epocas=None):
    cols = RESPUESTAS[estrategia]
    X_tr, Y_tr = entrada(W_TODO[filas_tr]), datos[cols].to_numpy()[filas_tr]
    X_va = Y_va = None
    if filas_va is not None:
        X_va, Y_va = entrada(W_TODO[filas_va]), datos[cols].to_numpy()[filas_va]
    grupos = [[j] for j in range(len(cols))] if estrategia == 'directa_multi' else [list(range(len(cols)))]
    modelos, epocas_usadas = [], []
    for g in grupos:
        predecir, e = ajustar(params, X_tr, Y_tr[:, g], X_va, None if Y_va is None else Y_va[:, g], epocas)
        modelos.append(predecir)
        epocas_usadas.append(e)
    W = W_TODO[filas_ev]
    if estrategia == 'recursiva':
        pred = pronostico_recursivo(modelos[0], W, entrada)
    else:
        pred = np.column_stack([predecir(entrada(W)) for predecir in modelos])
    e = None if None in epocas_usadas else int(round(np.mean(epocas_usadas)))
    return a_niveles(pred, filas_ev, estrategia), e

def recm_pliegues(entrada, ajustar, params, estrategia, esquema, epocas=None):
    recm, epocas_usadas = [], []
    for f_tr, f_va in pliegues(esquema, estrategia):
        P, e = pronosticar(entrada, ajustar, params, estrategia, f_tr, f_va, filas_va=f_va, epocas=epocas)
        recm.append(recm_h6(P, f_va))
        epocas_usadas.append(e)
    return recm, epocas_usadas

def recm_cv(recm):
    return float(np.sqrt(np.mean(np.square(recm))))

def combinaciones(espacio):
    claves = list(espacio)
    return [dict(zip(claves, valores)) for valores in itertools.product(*espacio.values())]

def sortear(espacio, n):
    todas = combinaciones(espacio)
    return random.Random(SEMILLA).sample(todas, min(n, len(todas)))

def buscar(entrada, ajustar, candidatos, rondas, estrategia, esquema):
    historial = []
    for r, epocas in enumerate(rondas, start=1):
        evaluados = []
        for params in candidatos:
            recm, ep = recm_pliegues(entrada, ajustar, params, estrategia, esquema, epocas)
            e = None if None in ep else int(round(np.mean(ep)))
            evaluados.append((recm_cv(recm), params, e))
            historial.append({'ronda': r, 'épocas máx.': epocas, **params, 'RECM_cv (h=6)': recm_cv(recm), 'épocas usadas': e})
        evaluados.sort(key=lambda x: x[0])
        candidatos = [p for _, p, _ in evaluados[:max(1, len(evaluados) // 3)]]
    mejor_recm, mejor_params, mejor_epocas = evaluados[0]
    return {'params': mejor_params, 'epocas': mejor_epocas, 'recm_val': mejor_recm, 'historial': pd.DataFrame(historial)}

def correr_experimentos(nombre, entrada, ajustar, busquedas):
    filas_tabla, predicciones, historiales = [], {}, {}
    for estrategia in ESTRATEGIAS:
        for esquema in ESQUEMAS_CV:
            for metodo, (candidatos, rondas) in busquedas.items():
                inicio = time.time()
                r = buscar(entrada, ajustar, candidatos, rondas, estrategia, esquema)
                P, _ = pronosticar(entrada, ajustar, r['params'], estrategia,
                                   filas_entrenamiento(CORTE_TEST, estrategia), FILAS_PRUEBA, epocas=r['epocas'])
                clave = (nombre, estrategia, esquema, metodo)
                predicciones[clave], historiales[clave] = P, r['historial']
                filas_tabla.append({
                    'Modelo': nombre, 'Estrategia': estrategia, 'Validación': esquema, 'Búsqueda': metodo,
                    'Hiperparámetros': r['params'], 'Épocas': r['epocas'],
                    'RECM val (h=6)': r['recm_val'], 'RECM test (h=6)': recm_h6(P, FILAS_PRUEBA),
                    'Segundos': time.time() - inicio
                })
                print(f"{nombre:5s} | {estrategia:13s} | {esquema:10s} | {metodo:9s} | "
                      f"val={r['recm_val']:.3f} | test={filas_tabla[-1]['RECM test (h=6)']:.3f} | {time.time() - inicio:.0f}s", flush=True)
    return {'tabla': pd.DataFrame(filas_tabla), 'predicciones': predicciones, 'historiales': historiales}

def elegir_por_validacion(tabla, grupos):
    return tabla.loc[tabla.groupby(grupos)['RECM val (h=6)'].idxmin()]

# 5. Árbol de decisión

In [ ]:
def entrada_arbol(W):
    rezagos = W[:, [VENTANA - k for k in LAGS_MODELO], 0]
    dummies = (W[:, -1, 1][:, None] == np.arange(2, 13)).astype(float)
    return np.column_stack([rezagos, dummies])

def ajustar_arbol(params, X_tr, y_tr, X_va=None, y_va=None, epocas=None):
    arbol = DecisionTreeRegressor(**params, random_state=SEMILLA)
    arbol.fit(X_tr, y_tr if y_tr.shape[1] > 1 else y_tr.ravel())
    return (lambda X: arbol.predict(X).reshape(len(X), -1)), None

# Curva de validación para profundidad
curva = {}
for d in range(1, 11):
    curva[d] = {esquema: recm_cv(recm_pliegues(entrada_arbol, ajustar_arbol, {'max_depth': d}, 'recursiva', esquema)[0])
                for esquema in ESQUEMAS_CV}
curva = pd.DataFrame(curva).T
curva.plot(marker='o', title="Curva de validación Árbol (Recursivo)")
plt.ylabel("RECM_cv (h=6)"); plt.show()

In [ ]:
busquedas_arbol = {
    'grid':   (combinaciones({'max_depth': [2, 4, 6], 'min_samples_leaf': [1, 5, 10]}), [None]),
    'random': (sortear({'max_depth': list(range(2, 11)), 'min_samples_leaf': [1, 2, 5, 10, 20]}, 4), [None]),
}
res_arbol = correr_experimentos('Árbol', entrada_arbol, ajustar_arbol, busquedas_arbol)
res_arbol['tabla'].drop(columns=['Épocas']).round(4)

# 6. Perceptrón Multicapa (MLP)

In [ ]:
EPOCAS_MAX = 60
LOTE = 16
PACIENCIA = 5

def entrada_mlp(W):
    X = entrada_arbol(W)
    X[:, :len(LAGS_MODELO)] = a_z(X[:, :len(LAGS_MODELO)])
    return X.astype('float32')

def crear_mlp(capas=(8,), activacion="relu", tasa=1e-3, n_salidas=1):
    keras.utils.set_random_seed(SEMILLA)
    modelo = keras.Sequential([keras.Input(shape=(len(COLS_TABLA),))])
    for unidades in capas:
        modelo.add(keras.layers.Dense(unidades, activation=activacion))
    modelo.add(keras.layers.Dense(n_salidas))
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=tasa), loss="mse")
    return modelo

def parada_temprana(paciencia=5):
    return keras.callbacks.EarlyStopping(monitor="val_loss", patience=paciencia, restore_best_weights=True)

def entrenar_con_validacion(modelo, X_tr, y_tr, X_va, y_va, paciencia=5, epocas=EPOCAS_MAX, lote=LOTE):
    historia = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=epocas, batch_size=lote,
                          callbacks=[parada_temprana(paciencia)], verbose=0)
    historia = pd.DataFrame(historia.history)
    historia.index = historia.index + 1
    return historia

def ajustar_mlp(params, X_tr, y_tr, X_va=None, y_va=None, epocas=EPOCAS_MAX):
    p = dict(params)
    lote = p.pop('lote', LOTE)
    modelo = crear_mlp(**p, n_salidas=y_tr.shape[1])
    if X_va is not None:
        historia = entrenar_con_validacion(modelo, X_tr, a_z(y_tr), X_va, a_z(y_va), PACIENCIA, epocas, lote)
        epocas_usadas = int(historia["val_loss"].idxmin())
    else:
        modelo.fit(X_tr, a_z(y_tr), epochs=epocas, batch_size=lote, verbose=0)
        epocas_usadas = epocas
    return (lambda X: a_puntos(modelo.predict(X, verbose=0))), epocas_usadas

In [ ]:
ESPACIO_MLP = {'capas': [(8,), (16,), (32,), (16, 8), (32, 16)], 'activacion': ['relu', 'tanh'],
               'tasa': [1e-3, 3e-3, 1e-2], 'lote': [16, 32]}
busquedas_mlp = {
    'grid':      (combinaciones({'capas': [(16,), (32, 16)], 'activacion': ['relu'], 'tasa': [1e-2], 'lote': [16]}), [EPOCAS_MAX]),
    'random':    (sortear(ESPACIO_MLP, 2), [EPOCAS_MAX]),
    'hyperband': (sortear(ESPACIO_MLP, 4), [10, EPOCAS_MAX]),
}
res_mlp = correr_experimentos('MLP', entrada_mlp, ajustar_mlp, busquedas_mlp)
res_mlp['tabla'].drop(columns=['Segundos']).round(4)

# 7. Red LSTM

In [ ]:
def entrada_lstm(W):
    angulo = 2 * np.pi * W[:, :, 1] / 12
    return np.stack([a_z(W[:, :, 0]), np.sin(angulo), np.cos(angulo)], axis=-1).astype('float32')

def crear_lstm(capas=(8,), tasa=1e-3, n_salidas=1):
    keras.utils.set_random_seed(SEMILLA)
    modelo = keras.Sequential([keras.Input(shape=(VENTANA, 3))])
    for j, unidades in enumerate(capas):
        modelo.add(keras.layers.LSTM(unidades, return_sequences=(j < len(capas) - 1)))
    modelo.add(keras.layers.Dense(n_salidas))
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=tasa), loss="mse")
    return modelo

def ajustar_lstm(params, X_tr, y_tr, X_va=None, y_va=None, epocas=EPOCAS_MAX):
    p = dict(params)
    lote = p.pop('lote', LOTE)
    modelo = crear_lstm(**p, n_salidas=y_tr.shape[1])
    if X_va is not None:
        historia = entrenar_con_validacion(modelo, X_tr, a_z(y_tr), X_va, a_z(y_va), PACIENCIA, epocas, lote)
        epocas_usadas = int(historia["val_loss"].idxmin())
    else:
        modelo.fit(X_tr, a_z(y_tr), epochs=epocas, batch_size=lote, verbose=0)
        epocas_usadas = epocas
    return (lambda X: a_puntos(modelo.predict(X, verbose=0))), epocas_usadas

In [ ]:
ESPACIO_LSTM = {'capas': [(8,), (16,), (32,), (16, 8)], 'tasa': [1e-3, 3e-3, 1e-2], 'lote': [16, 32]}
busquedas_lstm = {
    'grid':      (combinaciones({'capas': [(16,), (32,)], 'tasa': [1e-2], 'lote': [16]}), [EPOCAS_MAX]),
    'random':    (sortear(ESPACIO_LSTM, 2), [EPOCAS_MAX]),
    'hyperband': (sortear(ESPACIO_LSTM, 4), [10, EPOCAS_MAX]),
}
res_lstm = correr_experimentos('LSTM', entrada_lstm, ajustar_lstm, busquedas_lstm)
res_lstm['tabla'].drop(columns=['Segundos']).round(4)

# 8. Comparación final

In [ ]:
tabla_total = pd.concat([res_arbol['tabla'], res_mlp['tabla'], res_lstm['tabla']], ignore_index=True)
predicciones_total = {**res_arbol['predicciones'], **res_mlp['predicciones'], **res_lstm['predicciones']}
tabla_total['Mejora vs persistencia (%)'] = 100 * (1 - tabla_total['RECM test (h=6)'] / RECM_PERSISTENCIA)

columnas_resumen = ['Modelo', 'Validación', 'Estrategia', 'Búsqueda', 'Hiperparámetros', 'Épocas',
                    'RECM val (h=6)', 'RECM test (h=6)', 'Mejora vs persistencia (%)']
ganadores = elegir_por_validacion(tabla_total, ['Modelo', 'Validación'])[columnas_resumen].sort_values(['Validación', 'RECM val (h=6)'])
display(ganadores.round(4))

In [ ]:
ganadores_sec = (elegir_por_validacion(tabla_total[tabla_total['Validación'] == 'secuencial'], ['Modelo'])
                 .set_index('Modelo'))
colores_modelo = {'Árbol': '#27ae60', 'MLP': '#2980b9', 'LSTM': '#c0392b'}

mes_6 = datos.index[FILAS_PRUEBA] + pd.DateOffset(months=H - 1)
plt.figure(figsize=(11, 4.5))
plt.plot(mes_6, niveles_reales(FILAS_PRUEBA)[:, -1], color='black', marker='o', markersize=3, label='Real (h=6)')
for modelo, g in ganadores_sec.iterrows():
    P = predicciones_total[(modelo, g['Estrategia'], 'secuencial', g['Búsqueda'])]
    plt.plot(mes_6, P[:, -1], color=colores_modelo[modelo], ls='--',
             label=f"{modelo} ({g['Estrategia']}) RECM={g['RECM test (h=6)']:.3f}")
plt.plot(mes_6, P_PERSISTENCIA[:, -1], color='gray', ls=':', label=f'Persistencia RECM={RECM_PERSISTENCIA:.3f}')
plt.title("Evaluación en Prueba a 6 meses: ganadores con Validación Secuencial", fontweight='bold')
plt.ylabel("Puntos del ISE")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Error por horizonte h=1 ... h=6
reales_prueba = niveles_reales(FILAS_PRUEBA)
def recm_por_horizonte(P):
    return [evaluar(reales_prueba[:, h], P[:, h])["RECM"] for h in range(H)]

con_trayectoria = tabla_total[(tabla_total['Validación'] == 'secuencial') & (tabla_total['Estrategia'] != 'directa_h')]
mejores_tray = elegir_por_validacion(con_trayectoria, ['Modelo']).set_index('Modelo')
recm_horizonte = pd.DataFrame({'Persistencia': recm_por_horizonte(P_PERSISTENCIA)}, index=pd.Index(range(1, H + 1), name='h'))
for modelo, g in mejores_tray.iterrows():
    P = predicciones_total[(modelo, g['Estrategia'], 'secuencial', g['Búsqueda'])]
    recm_horizonte[f"{modelo} ({g['Estrategia']})"] = recm_por_horizonte(P)

display(recm_horizonte.round(4))
recm_horizonte.plot(marker='o', figsize=(8, 3.8), title="RECM en prueba por horizonte (1 a 6 meses)")
plt.ylabel("RECM (puntos ISE)")
plt.tight_layout()
plt.show()

# 9. Producción: pronóstico de julio a diciembre de 2026

In [ ]:
funciones_modelo = {'Árbol': (entrada_arbol, ajustar_arbol), 'MLP': (entrada_mlp, ajustar_mlp), 'LSTM': (entrada_lstm, ajustar_lstm)}
fila_produccion = [len(datos) - 1]
pronosticos_prod = {}

for modelo, g in ganadores_sec.iterrows():
    entrada, ajustar = funciones_modelo[modelo]
    epocas = None if pd.isna(g['Épocas']) else int(g['Épocas'])
    P, _ = pronosticar(entrada, ajustar, g['Hiperparámetros'], g['Estrategia'],
                       filas_entrenamiento(FECHAS_EXT[-1], g['Estrategia']), fila_produccion, epocas=epocas)
    pronosticos_prod[f"{modelo} ({g['Estrategia']})"] = P[0]

fechas_futuras = pd.date_range(start=FECHAS[-1] + pd.DateOffset(months=1), periods=H, freq=FECHAS.freq or 'MS')
tabla_prod = pd.DataFrame(pronosticos_prod, index=fechas_futuras.strftime('%Y-%m')).round(2)
tabla_prod.index.name = 'Mes pronosticado'

print(f"Último dato real conocido ({FECHAS[-1]:%Y-%m}): {Y[-1]:.2f} puntos")
display(tabla_prod)

plt.figure(figsize=(11, 4.5))
plt.plot(FECHAS[-36:], Y[-36:], color='black', marker='o', markersize=3, label='Histórico reciente')
for (nombre, trayectoria), color in zip(pronosticos_prod.items(), colores_modelo.values()):
    mascara = ~np.isnan(trayectoria)
    plt.plot(fechas_futuras[mascara], trayectoria[mascara], color=color, ls='--', marker='s', markersize=4, label=nombre)
plt.title("Pronóstico en producción para el segundo semestre de 2026", fontweight='bold')
plt.ylabel("Puntos del ISE")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# 10. Conclusiones

1. **Evaluación de tendencia:** La comparación de las 6 técnicas de tendencia demostró que mientras los filtros y splines logran residuos pequeños en muestra, sufren de severos problemas de borde y requieren modelos auxiliares para proyectarse hacia el futuro. La **diferenciación de primer orden** resuelve este dilema limpiando la tendencia y permitiendo la reconstrucción exacta por suma telescópica.
2. **Multi-estrategia:** Las estrategias recursiva y MIMO (*one-shot*) permiten reconstruir la trayectoria completa sin discontinuidades.
3. **Validación secuencial:** Promediar bloques temporales consecutivos previene que el entrenamiento sea sesgado por choques aislados (como la crisis de 2020), ofreciendo un criterio de selección de hiperparámetros más robusto que la validación simple.

# 11. Pendientes
1. Repetir la extracción de tendencia usando un modelo híbrido ARIMA-Splines para comparar la suma telescópica contra la desestacionalización aditiva.
2. Incorporar covariables macroeconómicas exógenas (inflación, tasa de intervención, desempleo) en el vector de entrada de la LSTM.
3. Evaluar intervalos de predicción basados en remuestreo de residuales (*bootstrap* de bloques).